# NYC Taxi Trip Duration — Baselines

Loads and cleans data via `preprocess.py`, splits by pickup time (train before 2016-06-13, validate after), and compares baseline predictors on `log1p(trip_duration)`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

from preprocess import (
    load_train,
    clean_dataframe,
    quality_check_summary,
    build_features,
    add_log_target,
    train_val_split_by_date,
    evaluate_predictions,
    DEFAULT_VAL_START,
)

## Load and clean

In [ ]:
raw = load_train()
print(f"Raw train rows: {len(raw):,}")
quality_check_summary(raw)

In [ ]:
df = clean_dataframe(raw)
df = add_log_target(df)
print(f"Clean rows: {len(df):,} ({100 * len(df) / len(raw):.2f}% of raw)")
df[["trip_duration", "log_trip_duration", "manhattan_km"]].describe()

## Time-based train / validation split

In [ ]:
train_df, val_df = train_val_split_by_date(df, val_start=DEFAULT_VAL_START)
print(f"Train: {len(train_df):,}  |  Val (pickup >= {DEFAULT_VAL_START}): {len(val_df):,}")
print(f"Train pickups: {train_df['pickup_datetime'].min()} → {train_df['pickup_datetime'].max()}")
print(f"Val pickups:   {val_df['pickup_datetime'].min()} → {val_df['pickup_datetime'].max()}")

In [ ]:
X_train = build_features(train_df)
X_val = build_features(val_df)
y_train = train_df["trip_duration"].to_numpy()
y_val = val_df["trip_duration"].to_numpy()
y_train_log = train_df["log_trip_duration"].to_numpy()
y_val_log = val_df["log_trip_duration"].to_numpy()

X_train.shape, X_val.shape

## Baselines

| Model | Idea |
|-------|------|
| Mean log duration | Predict constant `mean(log1p(duration))` on train |
| Median seconds | Predict global median duration |
| Distance + time linear | `LinearRegression` on engineered features |
| Fixed speed | `duration ≈ manhattan_km / speed` with speed fit on train |

In [ ]:
def run_baseline(name: str, y_pred: np.ndarray) -> dict:
    metrics = evaluate_predictions(y_val, y_pred)
    return {"model": name, **metrics}


results = []

# 1) Mean of log target (Kaggle-style constant)
mean_log = float(y_train_log.mean())
pred_mean_log = np.expm1(np.full_like(y_val, mean_log, dtype=float))
results.append(run_baseline("mean_log_duration", pred_mean_log))

# 2) Global median duration (seconds)
median_sec = float(np.median(y_train))
pred_median = np.full_like(y_val, median_sec, dtype=float)
results.append(run_baseline("median_duration_sec", pred_median))

# 3) Linear regression on features (fit in log space)
lin = LinearRegression()
lin.fit(X_train, y_train_log)
pred_lin_log = lin.predict(X_val)
pred_lin_sec = np.expm1(pred_lin_log)
results.append(run_baseline("linear_distance_time", pred_lin_sec))

# 4) Fixed speed from train (km/h → seconds)
train_speed = (
    train_df["manhattan_km"] / (train_df["trip_duration"] / 3600)
).median()
pred_speed = (X_val["manhattan_km"] / train_speed) * 3600
results.append(run_baseline(f"fixed_speed_{train_speed:.1f}kmh", pred_speed))

results_df = pd.DataFrame(results).sort_values("rmse_log")
results_df

**Primary metric:** `rmse_log` (RMSE on `log1p(duration)`). Lower is better.

`rmse_sec` / `mae_sec` are on raw seconds after back-transform (or median/mean in seconds).

Next steps: gradient boosting on the same features + time split; optional `sample` argument for faster iteration during dev.